In [ ]:
import os
from tqdm import tqdm
import pandas as pd

from stanza_tokenizer import StanzaTokenizer

In [ ]:
tokenizer = StanzaTokenizer()

In [ ]:
path = 'sentence_dataframe.tsv'
df = pd.read_csv(path, sep='\t')
len(df), df.columns

In [ ]:
# save_path = f'token_data/token_data_all.tsv'
# assert(os.path.exists(os.path.dirname(save_path))), f"Directory does not exist: {os.path.dirname(save_path)}"

In [ ]:
def sanity_check(text):
    assert(text.lower() == text), f"Text is not lowercase: {text}"
    assert(text.strip() == text), f"Text is not stripped: '{text}'"
    bad_chars = {
        "“": '"',
        "”": '"',
        "’": "'",
    }
    for bad_c, good_c in bad_chars.items():
        assert(bad_c not in text), f"Text contains bad character {bad_c}: {text}"

In [ ]:
chunk_size = 20000
# chunk_size = 20
if len(df) % chunk_size == 0:
    n = len(df) // chunk_size
else:
    n = len(df) // chunk_size + 1
for chunk_i in range(n):

    save_path = 'token_chunks/token_data_chunk_' + str(chunk_i) + '.tsv'

    if os.path.exists(save_path):
        print(f"{save_path} already exists, skipping...")
        continue

    start_index = chunk_i*chunk_size
    end_index = (chunk_i+1)*chunk_size
    chunk_df = df.iloc[start_index:end_index]

    print(f"Processing chunk {chunk_i+1}/{n}, rows {start_index} to {end_index}...")

    ### Token Step ###
    data = {}
    for _, row in tqdm(chunk_df.iterrows(), total=len(chunk_df)):
        # normalize text
        text = row['text_it']
        text = text.replace("’", "'")
        text = text.replace("“", '"').replace("”", '"')
        text = text.strip()

        tokens = tokenizer.tokenize(text)
        for token in tokens:
            text = token['text']
            sanity_check(text)
            lemma = token['lemma']
            lemma = lemma.lower()
            parts_lemma = token['parts_lemma']
            token_hash = token['token_hash']
            group_hash = token['group_hash']
            if token_hash not in data:
                data[token_hash] = {
                    'text': text,
                    'lemma': lemma,
                    'parts_lemma': parts_lemma,
                    'pos': token['pos'],
                    'parts_pos': token['parts_pos'],
                    'xpos': token['xpos'],
                    'deprel': token['deprel'],
                    'feats': token['feats'],
                    'feats': token['feats'],
                    'count': 0,
                    'sentences': set(),
                    'group_hash': group_hash,
                }
            data[token_hash]['count'] += 1
            if len(data[token_hash]['sentences']) < 20:
                data[token_hash]['sentences'].add(row['hash'])

    ### Dataframe Step ###
    cols = list(data[list(data.keys())[0]].keys()) + ['token_hash']
    token_df_data = {c: [] for c in cols}
    for token_hash, info in data.items():
        if info['pos'] == 'PUNCT' or info['pos'] == "PROPN":
            continue
        for col in cols:
            if col == 'token_hash':
                token_df_data[col].append(token_hash)
            elif col == 'sentences':
                token_df_data[col].append(list(info[col]))
            else:
                token_df_data[col].append(info[col])
    token_df = pd.DataFrame(token_df_data)
    token_df.sort_values(by='count', ascending=False, inplace=True)

    if not os.path.exists(os.path.dirname(save_path)):
        os.makedirs(os.path.dirname(save_path))
    token_df.to_csv(save_path, sep='\t', index=False)

    del data, token_df, token_df_data